### Transform Circuits Data


In [0]:
%run ../00-common/01-environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

#### Step 1 - Read curcuits bronze table 

In [0]:
# circuits_df = spark.read.table(bronze_table) DataFram Reader API, give adv like time travel and versioning with .option()

circuits_df = spark.table(bronze_table) 

display(circuits_df)

#### 2. Keep Neccessory columns 

In [0]:
from pyspark.sql import functions as F

circuits_selected_df = circuits_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)

display(circuits_selected_df)

#### 3&4 Standardise Column Names

In [0]:
circuits_renamed_df = circuits_selected_df.withColumnsRenamed({
    "circuitId": "circuit_id",
    "circuitName": "circuit_name",
    "lat": "latitude",
    "long": "longitude",
})

display(circuits_renamed_df)

#### 5. Remove NUll Ids , keep valid rows 

In [0]:
circuits_valid_df = circuits_renamed_df.filter(F.col("circuit_id").isNotNull())

display(circuits_valid_df)

#### 6. Remove Duplicates

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(['circuit_id'])
display(circuits_distinct_df)

#### 7. Transform Values of Columns circuit_name, locality to Title Case

In [0]:
circuit_final_df = (
    circuits_distinct_df
    .withColumn('circuit_name', F.initcap(F.col('circuit_name')))
    .withColumn('locality', F.initcap(F.col('locality')))
)

display(circuit_final_df)

#### 8. Wirte the transform data to silver circuits table

In [0]:
(circuit_final_df
 .write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(silver_table)
)

In [0]:
display(spark.read.table(silver_table))